# Customer Risk Tier Generation\n\nBusiness logic:\n1. Blend credit quality (Score/PD/LGD), early warning pressure, AML risk, and digital behavior.\n2. Compute a composite risk index per customer.\n3. Assign High/Medium/Low tier for risk discipline and portfolio monitoring.

In [ ]:
from pyspark.sql import functions as F\n\ncustomers = spark.table(\"Retail_Customers\").select(\"CustomerID\", \"DigitalAdoptionScore\")\nscores = spark.table(\"Risk_CreditRiskScores\")\news = spark.table(\"Risk_EarlyWarningSignals\").groupBy(\"CustomerID\").agg(F.count(\"*\").alias(\"EWSCount\"))\naml = spark.table(\"AML_Alerts\").groupBy(\"CustomerID\").agg(F.avg(\"RiskScore\").alias(\"AvgAmlRisk\"), F.count(\"*\").alias(\"AmlAlertCount\"))\n\nrisk_tiers = (\n    customers.alias(\"c\")\n    .join(scores.alias(\"s\"), on=\"CustomerID\", how=\"left\")\n    .join(ews.alias(\"e\"), on=\"CustomerID\", how=\"left\")\n    .join(aml.alias(\"a\"), on=\"CustomerID\", how=\"left\")\n    .fillna({\"EWSCount\": 0, \"AvgAmlRisk\": 0.0, \"AmlAlertCount\": 0, \"PD\": 0.01, \"LGD\": 0.40, \"Score\": 700})\n    .withColumn(\"CreditRiskComponent\", ((850 - F.col(\"Score\")) / 550.0) * 45)\n    .withColumn(\"ProbabilityLossComponent\", (F.col(\"PD\") * F.col(\"LGD\")) * 100 * 25)\n    .withColumn(\"EarlyWarningComponent\", F.least(F.col(\"EWSCount\") * 3.5, F.lit(15.0)))\n    .withColumn(\"AmlComponent\", F.least((F.col(\"AvgAmlRisk\") / 100.0) * 10 + F.col(\"AmlAlertCount\") * 0.8, F.lit(10.0)))\n    .withColumn(\"DigitalRiskOffset\", (100 - F.col(\"DigitalAdoptionScore\")) / 100.0 * 5)\n    .withColumn(\"CompositeRiskScore\", F.round(F.col(\"CreditRiskComponent\") + F.col(\"ProbabilityLossComponent\") + F.col(\"EarlyWarningComponent\") + F.col(\"AmlComponent\") + F.col(\"DigitalRiskOffset\"), 2))\n    .withColumn(\"GeneratedAt\", F.current_timestamp())\n    .withColumn(\"RiskTier\", F.when(F.col(\"CompositeRiskScore\") >= 70, F.lit(\"High\")).when(F.col(\"CompositeRiskScore\") >= 40, F.lit(\"Medium\")).otherwise(F.lit(\"Low\")))\n    .select(\"CustomerID\", \"RiskTier\", \"CompositeRiskScore\", \"Score\", \"PD\", \"LGD\", \"EWSCount\", \"AvgAmlRisk\", \"AmlAlertCount\", \"DigitalAdoptionScore\", \"GeneratedAt\")\n)\n\n(\n    risk_tiers\n    .write.format(\"delta\")\n    .mode(\"overwrite\")\n    .option(\"overwriteSchema\", \"true\")\n    .saveAsTable(\"Risk_CustomerRiskTierDaily\")\n)\n\ndisplay(risk_tiers.orderBy(F.desc(\"CompositeRiskScore\")).limit(50))\n